# AI Application — Context Builder & Test
This notebook previews the pre-computed context that gets passed to the Groq LLM, and lets you test questions before launching the Streamlit app.

**To run the chat app:**
```bash
streamlit run app.py
```

In [3]:
import os
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from data_engine import load_data, build_context, build_system_prompt, ask_question

load_dotenv()

DATA_DIR = os.path.join(os.getcwd(), "data")

data = load_data(DATA_DIR)
print("All tables loaded:")
for key, df in data.items():
    print(f"  {key:20s} → {df.shape[0]:,} rows × {df.shape[1]} cols")

All tables loaded:
  account_dim          → 24 rows × 5 cols
  date_dim             → 518 rows × 6 cols
  hcp_dim              → 90 rows × 5 cols
  rep_dim              → 9 rows × 4 cols
  territory_dim        → 3 rows × 4 cols
  fact_rx              → 1,530 rows × 5 cols
  fact_rep_act         → 2,962 rows × 9 cols
  fact_ln              → 570 rows × 5 cols
  fact_payor           → 480 rows × 4 cols


## Preview the Pre-Computed Context
This is exactly what the LLM sees as its knowledge base.

In [4]:
context = build_context(data)
print(context)

=== DATASET OVERVIEW ===
Territories: 3 | Reps: 9 | HCPs: 90 | Accounts: 24
Date range: 2024-08-01 to 2025-12-31
Total Rx records: 1,530 | Total activity records: 2,962

=== FLAGGED INSIGHTS & ALERTS ===
  [WARN] Sage Brown is below average on Tier A coverage: 19.9% vs team avg 28.8%.
  [WARN] Jamie Thomas is below average on Tier A coverage: 23.1% vs team avg 28.8%.
  [WARN] River White is below average on Tier A coverage: 21.5% vs team avg 28.8%.
  [ALERT] Taylor Kim has a low call completion rate: 56.3% (team avg 60.3%).
  [ALERT] Taylor Wilson has a low call completion rate: 59.2% (team avg 60.3%).
  [ALERT] Morgan Chen has a low call completion rate: 58.3% (team avg 60.3%).
  [ALERT] Jamie Thomas has a low call completion rate: 57.6% (team avg 60.3%).
  [ALERT] Reese Miller has a low call completion rate: 59.6% (team avg 60.3%).
  [ALERT] River Miller has a low call completion rate: 59.8% (team avg 60.3%).
  [POSITIVE] GAZYVA NRx grew 24.3% in the most recent quarter.

=== BRANDS 

## Test a Question Against Groq

In [6]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
system_prompt = build_system_prompt(context)
history = []

q = "Which rep has the lowest Tier A HCP coverage and how does it compare to the top rep?"
print(f"Q: {q}\n")
answer = ask_question(client, system_prompt, history, q)
history.append({"role": "user", "content": q})
history.append({"role": "assistant", "content": answer})
print(f"A: {answer}")

Q: Which rep has the lowest Tier A HCP coverage and how does it compare to the top rep?

A: According to the REP SCORECARD, Sage Brown has the lowest Tier A HCP coverage at 19.9%. 

The top rep for Tier A HCP coverage is River Miller with 37.3% of their calls being to Tier A HCPs. 

Here's the comparison: 
- Sage Brown: 19.9% 
- River Miller: 37.3% 

This indicates a significant gap between the rep with the lowest Tier A coverage and the top rep. 
Actionable recommendation: Provide additional training and support to Sage Brown to improve their Tier A HCP coverage.


In [7]:
q2 = "Which territory does that rep belong to?"
print(f"Q: {q2}\n")
answer2 = ask_question(client, system_prompt, history, q2)
history.append({"role": "user", "content": q2})
history.append({"role": "assistant", "content": answer2})
print(f"A: {answer2}")

Q: Which territory does that rep belong to?

A: I don't have enough data to answer that — the dataset covers rep performance, territory performance, and other sales metrics, but it does not explicitly link individual reps to specific territories.
